# SurvFace 04. Official pgvector search matrix

공식 gallery의 `official_all` identity template을 만들고 registered/unmated probe 전체를 공식 순서대로 검색합니다. 정식 기본 행렬은 `origin_512`와 `pca_256` 각각의 exact/HNSW 네 조합입니다. PQ code는 pgvector 검색 대상이 아닙니다.

각 장시간 DB load/template/search 작업은 네 조합 전체를 합친 약 10% 경계에서만 로그를 출력합니다. 실제 전체 검색은 사용자가 이 노트북을 직접 실행할 때만 시작됩니다.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.database import create_database_engine, init_database, load_database_settings
from research.experiments import run_survface_official_search_matrix
from research.protocols import build_survface_official_protocol
from research.runtime import ProgressReporter, RunStore, resolve_active_run

MODE = 'real'
DATA_FRACTION = 1.0
SEED = 42
EXECUTE_STAGE = True
COMPRESSION_PROFILES = ("origin_512", "pca_256")
SEARCH_MODES = ("exact", "hnsw")
TOP_K = 20
PROBE_LIMIT = None
BATCH_SIZE = 256
FULL_RUN_ACKNOWLEDGEMENT = "SURVFACE_FULL_SEARCH"
RUN_ROOT = PROJECT_ROOT / "runs/survface"
RUN_DIR = resolve_active_run(
    RUN_ROOT,
    environment_variable="RONBUN_SURVFACE_RUN_DIR",
)
PROGRESS = ProgressReporter(
    "SurvFace 04 official search matrix",
    heartbeat_seconds=None,
    milestone_percent=10,
)
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR),
    "compression_profiles": COMPRESSION_PROFILES,
    "search_modes": SEARCH_MODES,
    "top_k": TOP_K,
    "probe_limit_per_role": PROBE_LIMIT,
    "gallery_enrollment_policy": "official_all",
    "gallery_enrollment_target": 0,
    "full_run_acknowledged": FULL_RUN_ACKNOWLEDGEMENT == "SURVFACE_FULL_SEARCH",
}
preflight

In [ ]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if TOP_K != 20:
        raise ValueError("SurvFace 공식 평가용 top_k는 20이어야 합니다.")
    if PROBE_LIMIT is None and FULL_RUN_ACKNOWLEDGEMENT != "SURVFACE_FULL_SEARCH":
        raise RuntimeError("전체 검색 전 FULL_RUN_ACKNOWLEDGEMENT를 정확히 입력하십시오.")

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts("01_official_arcface_embedding_extraction")
    if any(profile != "origin_512" for profile in COMPRESSION_PROFILES):
        run.verify_phase_artifacts("03_survface_compressed_materialization_and_index")
    manifest = pd.read_csv(PROJECT_ROOT / "data/interim/survface/official_manifest.csv")
    protocol = build_survface_official_protocol(manifest)
    if not protocol.known_unknown_probes.empty:
        raise ValueError("SurvFace 공식 검색에 known_unknown이 들어갔습니다.")
    engine = create_database_engine(load_database_settings())
    init_database(engine)

    with run.phase("04_official_probe_search") as phase:
        suffix = f"A{phase.attempt:03d}"
        output_path = phase.attempt_dir / f"official_top20_search_matrix_{suffix}.csv"
        summary = run_survface_official_search_matrix(
            engine,
            run_uid=run.run_id,
            manifest=manifest,
            compression_profiles=COMPRESSION_PROFILES,
            search_modes=SEARCH_MODES,
            top_k=TOP_K,
            enrollment_policy="official_all",
            enrollment_target=0,
            output_path=output_path,
            batch_size=BATCH_SIZE,
            probe_limit_per_role=PROBE_LIMIT,
            progress=PROGRESS.callback(key_prefix=f"{run.run_id}:"),
        )
        frame = pd.read_csv(output_path)
        required = {
            "probe_type", "protocol_index", "query_identity_id",
            "ranked_identities", "ranked_distances", "compression_profile",
            "search_mode", "official_complete",
        }
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"search 결과 필수 열 누락: {sorted(missing)}")
        expected_groups = {
            (profile, mode)
            for profile in COMPRESSION_PROFILES
            for mode in SEARCH_MODES
        }
        actual_groups = set(zip(frame["compression_profile"], frame["search_mode"]))
        if actual_groups != expected_groups:
            raise ValueError(f"search matrix 불일치: {actual_groups} != {expected_groups}")
        lengths = frame["ranked_identities"].map(lambda value: len(json.loads(value)))
        if not lengths.eq(TOP_K).all():
            raise ValueError("모든 검색 행이 top-20을 포함해야 합니다.")

        summary_path = phase.attempt_dir / f"official_search_summary_{suffix}.json"
        summary_path.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        phase.publish_artifact(output_path)
        phase.publish_artifact(summary_path)
        phase.record_counts(
            rows=len(frame),
            combinations=len(expected_groups),
            registered=len(protocol.registered_probes) * len(expected_groups),
            unknown_unknown=len(protocol.unknown_unknown_probes) * len(expected_groups),
        )
    result = {"status": "completed", "run_id": run.run_id, **summary}
result

## 다음 단계

`combination_count=4`, `all_official_complete=true`, `known_unknown_count=0`을 확인한 뒤 `01_official_evaluation_and_visualization.ipynb`를 실행합니다.